In [4]:
import pandas as pd
from pathlib import Path

base_path = Path(r"C:\Users\btigr\Documents\UAO\5\ETL\ETL_2026_1\lab_04\lab_04_dq-main\lab_04_dq-main")

transformed_file = base_path / "data" / "processed" / "retail_transformed.csv"
dims_test = base_path / "data" / "dims_test" / "dim_test.csv"

# Dim date
def create_dim_date(df):

    dim_date = df[["invoice_date"]].drop_duplicates().reset_index(drop=True)

    dim_date["invoice_date"] = pd.to_datetime(dim_date["invoice_date"])

    dim_date["invoice_date_key"]  = dim_date.index + 1

    dim_date["full_date"] = dim_date["invoice_date"]
    dim_date["year"]      = dim_date["invoice_date"].dt.year
    dim_date["month"]     = dim_date["invoice_date"].dt.month
    dim_date["day"]       = dim_date["invoice_date"].dt.day

    dim_date = dim_date[["invoice_date_key", "full_date", "year", "month", "day"]]

    return dim_date


# Dim product
def create_dim_product(df):

    dim_product = df[["product", "price"]].drop_duplicates().reset_index(drop=True)

    dim_product["product_key"] = dim_product.index + 1

    dim_product = dim_product[["product_key", "product", "price"]]

    return dim_product


# Dim customer
def create_dim_customer(df):

    dim_customer = df[["customer_id"]].drop_duplicates().reset_index(drop=True)

    dim_customer["customer_key"]  = dim_customer.index + 1

    dim_customer = dim_customer[["customer_key", "customer_id"]]

    return dim_customer


# Dim location
def create_dim_location(df):

    dim_location = df[["country"]].drop_duplicates().reset_index(drop=True)

    dim_location["location_key"] = dim_location.index + 1

    dim_location = dim_location[["location_key", "country"]]

    return dim_location


# Fact table
def create_fact_sales(df, dim_product, dim_customer, dim_location, dim_date):

    df["invoice_date"] = pd.to_datetime(df["invoice_date"])

    # Merge para traer los keys
    df = df.merge(dim_product, on=["product", "price"], how="left")
    df = df.merge(dim_location, on="country",     how="left")
    df = df.merge(dim_customer, on="customer_id", how="left")

    # Para dim_date el merge debe ser por full_date
    dim_date_merge = dim_date[["invoice_date_key", "full_date"]].copy()
    dim_date_merge["full_date"] = pd.to_datetime(dim_date_merge["full_date"])
    df = df.merge(dim_date_merge, left_on="invoice_date", right_on="full_date", how="left")

    fact_sales = df[[
        "invoice_id",
        "invoice_date_key",
        "location_key",
        "customer_key",
        "product_key",
        "quantity",
        "total_revenue",
        "revenue_bin"
    ]].copy()

    return fact_sales

# Transform
def transform_data_DM(df):

    dim_date     = create_dim_date(df)
    dim_product  = create_dim_product(df)
    dim_customer = create_dim_customer(df)
    dim_location = create_dim_location(df)

    fact_sales = create_fact_sales(
        df,
        dim_product,
        dim_customer,
        dim_location,
        dim_date
    )

    return {
        "dim_date":     dim_date,
        "dim_product":  dim_product,
        "dim_customer": dim_customer,
        "dim_location": dim_location,
        "fact_sales":   fact_sales
    }

# Cargar el CSV
df = pd.read_csv(transformed_file)

# Ejecutar el transform
tablas = transform_data_DM(df)

print("=== dim_date ===")
display(tablas["dim_date"].head())

print("=== dim_product ===")
display(tablas["dim_product"].head())

print("=== dim_customer ===")
display(tablas["dim_customer"].head())

print("=== dim_location ===")
display(tablas["dim_location"].head())

print("=== fact_sales ===")
display(tablas["fact_sales"].head())

=== dim_date ===


,invoice_date_key,full_date,year,month,day
0,1,2023-03-25,2023,3,25
1,2,2023-11-23,2023,11,23
2,3,2023-05-13,2023,5,13
3,4,2023-05-28,2023,5,28
4,5,2023-11-20,2023,11,20


=== dim_product ===


,product_key,product,price
0,1,Mouse,525.96
1,2,Printer,74.92
2,3,Monitor,576.56
3,4,Mouse,312.70
4,5,Phone,400.61


=== dim_customer ===


,customer_key,customer_id
0,1,1102
1,2,1435
2,3,1348
3,4,1106
4,5,1071


=== dim_location ===


,location_key,country
0,1,CO
1,2,CL
2,3,EC
3,4,PE


=== fact_sales ===


,invoice_id,invoice_date_key,location_key,customer_key,product_key,quantity,total_revenue,revenue_bin
0,22951,1,1,1,1,2,1051.92,Low
1,21238,2,1,2,2,6,449.52,Low
2,29165,3,1,3,3,1,576.56,Low
3,23965,4,2,4,4,7,2188.90,Medium
4,25686,5,1,5,5,6,2403.66,Medium
